In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
from sklearn.preprocessing import LabelEncoder
import plotly.express as px
import io
df = pd.read_csv('C:/Users/jesus/Proyecto jupyter/04-default_credit.csv')

In [2]:
from ipywidgets import (
    VBox, HBox, Output,
    RadioButtons, Dropdown,
    FloatRangeSlider, IntSlider, Checkbox
)

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# =========================
# VARIABLES
# =========================
num_vars = df.select_dtypes('number').columns.tolist()

# =========================
# WIDGETS
# =========================
grafico = RadioButtons(
    options=[
        ('Histograma', 'hist'),
        ('Boxplot', 'box'),
        ('Scatter', 'scatter')
    ],
    description='Gráfico'
)

var1 = Dropdown(options=num_vars, description='Var 1')
var2 = Dropdown(options=num_vars, description='Var 2')

bins = IntSlider(min=5, max=50, step=5, value=10, description='Bins')

rango = FloatRangeSlider(description='Rango')

percentil = IntSlider(min=50, max=100, step=5, value=100, description='Percentil')

usar_rango = Checkbox(False, description='Usar rango')
usar_percentil = Checkbox(True, description='Quitar outliers')
mostrar_boxplot = Checkbox(True, description='Mostrar boxplot')

out = Output()

# =========================
# RANGO DINÁMICO
# =========================
def actualizar_rango(*args):
    col = df[var1.value].replace([np.inf, -np.inf], np.nan).dropna()
    rango.min = float(col.min())
    rango.max = float(col.max())
    rango.value = (rango.min, rango.max)

var1.observe(actualizar_rango, names='value')
actualizar_rango()

# =========================
# GRÁFICOS SEGUROS
# =========================
def graficar(*args):

    with out:
        out.clear_output()

        data = df.copy()

        v1 = var1.value
        v2 = var2.value

        # limpiar datos peligrosos
        data = data.replace([np.inf, -np.inf], np.nan).dropna()

        # =========================
        # FILTRO RANGO
        # =========================
        if usar_rango.value:
            data = data[(data[v1] >= rango.value[0]) &
                        (data[v1] <= rango.value[1])]

        # =========================
        # FILTRO PERCENTIL
        # =========================
        if usar_percentil.value and len(data) > 0:
            lim = data[v1].quantile(percentil.value / 100)
            data = data[data[v1] <= lim]

        if len(data) == 0:
            print("⚠️ No hay datos después de los filtros")
            return

        plt.figure()

        # =========================
        # HISTOGRAMA
        # =========================
        if grafico.value == 'hist':

            sns.histplot(data[v1], bins=bins.value, kde=True)

            if mostrar_boxplot.value:
                plt.figure()
                sns.boxplot(y=data[v1])

        # =========================
        # BOXPLOT (SEGURO)
        # =========================
        elif grafico.value == 'box':

            # evitar variables iguales
            if v1 == v2:
                v2 = v1

            temp = data[[v1, v2]].copy()
            temp[v1] = pd.to_numeric(temp[v1], errors='coerce')
            temp[v2] = pd.to_numeric(temp[v2], errors='coerce')
            temp = temp.dropna()

            if len(temp) > 0:
                sns.boxplot(x=temp[v1], y=temp[v2])
            else:
                print("⚠️ No hay datos válidos para boxplot")

        # =========================
        # SCATTER (SEGURO)
        # =========================
        elif grafico.value == 'scatter':

            temp = data[[v1, v2]].dropna()

            if len(temp) > 0:
                sns.scatterplot(x=temp[v1], y=temp[v2], alpha=0.5)
            else:
                print("⚠️ No hay datos válidos para scatter")

        plt.show()

# =========================
# CONEXIÓN
# =========================
def conectar(_=None):
    graficar()

for w in [
    grafico, var1, var2, bins,
    rango, percentil,
    usar_rango, usar_percentil, mostrar_boxplot
]:
    w.observe(conectar, names='value')

# =========================
# UI FINAL
# =========================
ui = VBox([
    grafico,
    HBox([var1, var2]),
    bins,
    rango,
    percentil,
    HBox([usar_rango, usar_percentil, mostrar_boxplot])
])

from IPython.display import display
display(ui, out)

actualizar_rango()
graficar()

Output()